# Data Merging — Stage 4 Assembly 02: Assemble Aggregate

## Input
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/{5 aggregate table names}.parquet` (from notebook 01)
- `lib.config` — `OUT`, `META`, `BINARIES`, `START_DATE`, `END_DATE`, `CLIP`, `DTYPE`, `FFILL_LIMIT`, `ASOF_TOL`

## Purpose
Builds the two final model-ready aggregate feature tables — a means-only table and a full-moments table — by combining the daily, weekly, and monthly aggregate tables onto a single daily trading calendar. This notebook is the aggregate-side analogue of what will presumably be a parallel panel-assembly notebook; it does not touch the stock-level panel tables at all.

## Step 1 — Gap Repair at Native Cadence
Before any cross-frequency merging happens, each of the five unioned tables (`daily_means`, `daily_full_moments`, `weekly`, `month_means`, `month_full_moments`) has short gaps forward-filled **at its own native frequency**, via `repair()`.

Three reasons this happens *before* expanding to daily rather than after, stated explicitly:
1. **The fill limit is literal in native units.** "3 months" means `limit=3` on the monthly table directly — there's no unit conversion needed to trading days.
2. **`merge_asof` matches rows, not values.** If a `NaN` gap were repaired only after expanding a weekly table to daily via `merge_asof`, that single missing weekly value would still propagate as five separate daily `NaN`s (one gap becomes five copies of the same gap) — repairing before expansion fixes it once.
3. **Cheaper.** Filling 3,107 weekly rows is less work than filling the ~5,234 daily rows that expansion would produce.

Note that `ffill` cannot repair **leading** `NaN`s (a gap at the very start of a series, before any real value has ever appeared) — that class of missingness is instead handled downstream by the `START_DATE` trim.

## Step 2 — Assemble: Daily Base + Weekly + Monthly onto the Daily Calendar
The `assemble()` function builds each output table by starting from the (gap-repaired) daily base table and layering weekly, then monthly data onto it via `pd.merge_asof(..., direction='backward')` — deliberately **not** a left merge, since a left merge would silently drop any source row whose date isn't already present in the daily calendar, whereas `merge_asof` correctly carries the last-known value forward onto every daily row.

- A `tolerance` is passed to each `merge_asof` call, described as a **backstop only** — the native-cadence `ffill` in Step 1 has already repaired anything shorter than the fill limit, so the tolerance's real job is to prevent a daily row from being paired with a stale value from months earlier, in the case where a source series has genuinely stopped reporting altogether.
- Before merging the monthly table, `target_monthly_return` is explicitly dropped. Once merged backward onto a daily index, that column becomes "last completed month's market return" — which is a legitimate feature in its own right, but the notebook deliberately doesn't keep it here because a column literally named `target` sitting inside the feature block is "an accident waiting to happen." Every actual target variable is built cleanly in a later notebook (04).
- Asserts no column-name collisions occur between the daily base and each incoming source (`weekly`, `monthly`) before merging.
- After all merges, trims the result to `[START_DATE, END_DATE]`, reporting the row count before and after the trim.

Produces two assembled outputs: `agg_means` (from `agg_market_daily_means` + `weekly` + `agg_market_monthly_means`) and `agg_full` (from `agg_market_daily_full_moments` + `weekly` + `agg_market_monthly_full_moments`).

## Step 3 — Finalise: Two Files Per Pipeline
For each of the two assembled tables, `finalise()` produces **two parallel output files**:

- **`{name}_nan.parquet`** — saved first, with all `NaN`s left intact. This file *is* the missingness indicator: downstream code can recover exactly which cells were imputed by calling `.isna()` on it, with no need for separate indicator columns.
- **`{name}.parquet`** — the actual model-ready file: features are clipped to `±CLIP`, remaining `NaN`s are filled with `0.0` (binaries filled separately, since they're `0`/`1` already and clipping is a no-op for them), and everything is cast to `DTYPE` (float32).

**Clip-before-fill ordering is deliberate**, even though `0` already sits well inside `±5` so the mathematical order doesn't matter here: clipping first guarantees the eventual fill value is unambiguously interpretable as "the centre of the range," rather than something that happened to depend on fill-then-clip sequencing.

An assertion confirms zero `NaN`s survive the fill before saving. A per-feature report captures NaN count/percentage and clipped-cell count for every feature, with a `flag` column marking any feature above 5% imputed (`'IMPUTED >5%'`).

## Step 4 — Imputed Mass and Clipping Diagnostics
Three printed reports, not saved to file individually (they're derived from the combined `rep` dataframe which *is* saved):

1. **Imputed mass above 5%** — lists every feature flagged in Step 3, with the reasoning spelled out: after z-scoring, an imputed zero literally means "exactly the expanding mean," which is a fabricated value rather than an observed one. Above roughly 5% imputation, a spline evaluated near zero is reading mostly fabricated signal rather than real data.
2. **Distribution of imputed mass** — per dataset (`agg_means` / `agg_full_moments`), reports median/p95/max imputation percentage and count of features above 1% imputed.
3. **Most clipped features** — the 15 features with the highest count of cells that hit the `±CLIP` boundary, alongside their imputation percentage.

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/agg_means_nan.parquet`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/agg_means.parquet`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/agg_full_moments_nan.parquet`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/agg_full_moments.parquet`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/fill_report_aggregate.csv` — per-feature NaN/clip/imputation-flag report across both datasets.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.config import (OUT, META, BINARIES, START_DATE, END_DATE, CLIP,
                        DTYPE, FFILL_LIMIT, ASOF_TOL)

IN_DIR  = OUT / '01_unioned'
OUT_DIR = OUT / '02_assembled'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.width', 200)

def load(tag):
    return pd.read_parquet(IN_DIR / f'{tag}.parquet').sort_values('date').reset_index(drop=True)

# ── GAP REPAIR AT NATIVE CADENCE ─────────────────────────────────────────────
# Before expanding to daily, not after. Three reasons: the limit is literal
# ("3 months" is limit=3 on the monthly table, no conversion to trading days);
# merge_asof matches rows not values, so a NaN in the weekly table would become
# NaN across a whole week on the daily index -- five copies of one gap; and it
# is cheaper on 3,107 weekly rows than 5,234 daily ones.
#
# ffill cannot touch LEADING NaN. That block is handled by the START_DATE trim.

def repair(df, tag, freq):
    meta = [c for c in META[tag] + BINARIES if c in df.columns]
    feats = [c for c in df.columns if c not in meta]
    before = int(df[feats].isna().sum().sum())
    df[feats] = df[feats].ffill(limit=FFILL_LIMIT[freq])
    after = int(df[feats].isna().sum().sum())
    print(f'  {tag:<34} {freq:<8} limit={FFILL_LIMIT[freq]:<2}  '
          f'NaN {before:>8,} -> {after:>8,}  ({before - after:,} repaired)')
    return df

print('=' * 100)
print('GAP REPAIR - native cadence, up to 3 periods')
print('=' * 100)

daily_means  = repair(load('agg_market_daily_means'),          'agg_market_daily_means',          'daily')
daily_full   = repair(load('agg_market_daily_full_moments'),   'agg_market_daily_full_moments',   'daily')
weekly       = repair(load('weekly_raw'),                      'weekly_raw',                      'weekly')
month_means  = repair(load('agg_market_monthly_means'),        'agg_market_monthly_means',        'monthly')
month_full   = repair(load('agg_market_monthly_full_moments'), 'agg_market_monthly_full_moments', 'monthly')

GAP REPAIR - native cadence, up to 3 periods
  agg_market_daily_means             daily    limit=5   NaN   79,351 ->   79,330  (21 repaired)
  agg_market_daily_full_moments      daily    limit=5   NaN  234,810 ->  234,705  (105 repaired)
  weekly_raw                         weekly   limit=9   NaN   68,776 ->    9,835  (58,941 repaired)
  agg_market_monthly_means           monthly  limit=3   NaN    6,168 ->    6,168  (0 repaired)
  agg_market_monthly_full_moments    monthly  limit=3   NaN   20,400 ->   20,400  (0 repaired)


In [2]:
# The monthly tables carry target_monthly_return. Merged backward onto a daily
# index it becomes "last completed month's market return" -- a legitimate
# feature, but a column named `target` sitting in the feature block is an
# accident waiting to happen. Dropped; notebook 04 builds every target.

def assemble(daily, monthly, monthly_tag, label):
    """Daily base + weekly + monthly, expanded onto the daily trading calendar."""
    print(f'\n{"=" * 100}\n{label}\n{"=" * 100}')

    out = daily.copy()
    print(f'  daily base                     {out.shape[1] - 1:>5} cols   {len(out):>6,} rows')

    for src, tol, name in [(weekly, ASOF_TOL['weekly'], 'weekly'),
                           (monthly, ASOF_TOL['monthly'], 'monthly')]:
        s = src.copy()
        if 'target_monthly_return' in s.columns:
            s = s.drop(columns='target_monthly_return')

        clash = (set(s.columns) & set(out.columns)) - {'date'}
        assert not clash, f'{name}: column name collision -> {sorted(clash)}'

        # merge_asof, not a left merge: a left merge silently loses any source
        # row whose date is not in the daily calendar. The tolerance is a
        # backstop only -- the native-cadence ffill has already repaired
        # anything shorter -- and stops a daily row pairing with a value from
        # months earlier when a series genuinely stops reporting.
        out = pd.merge_asof(out, s, on='date', direction='backward',
                            tolerance=pd.Timedelta(tol))
        print(f'  + {name:<28} {out.shape[1] - 1:>5} cols   tolerance {tol}')

    n0 = len(out)
    out = out[(out['date'] >= START_DATE) & (out['date'] <= END_DATE)].reset_index(drop=True)
    print(f'  trimmed to {START_DATE}..{END_DATE}   {n0:,} -> {len(out):,} rows')
    print(f'    {out["date"].min().date()} .. {out["date"].max().date()}')
    return out

agg_means = assemble(daily_means, month_means, 'agg_market_monthly_means',
                     'ASSEMBLE - MEANS ONLY')
agg_full  = assemble(daily_full,  month_full,  'agg_market_monthly_full_moments',
                     'ASSEMBLE - FULL MOMENTS')


ASSEMBLE - MEANS ONLY
  daily base                       291 cols    5,234 rows
  + weekly                         323 cols   tolerance 25D
  + monthly                        580 cols   tolerance 100D
  trimmed to 2007-08-01..2024-12-31   5,234 -> 4,384 rows
    2007-08-01 .. 2024-12-30

ASSEMBLE - FULL MOMENTS
  daily base                       823 cols    5,234 rows
  + weekly                         855 cols   tolerance 25D
  + monthly                       1705 cols   tolerance 100D
  trimmed to 2007-08-01..2024-12-31   5,234 -> 4,384 rows
    2007-08-01 .. 2024-12-30


In [3]:
# TWO FILES PER PIPELINE.
#   *_nan.parquet   NaN intact. This IS the missingness indicator -- recover it
#                   at load time with .isna(), no extra columns needed.
#   *.parquet       clipped, zero-filled, float32. Model-ready.
#
# Clip before fill: 0 is inside +/-5 so the order is immaterial, but clipping
# first means the fill value is unambiguously "the centre of the range".

def finalise(df, name):
    print(f'\n{"=" * 100}\n{name}\n{"=" * 100}')

    meta = [c for c in ['date', 'target_daily_return'] if c in df.columns]
    binaries = [c for c in BINARIES if c in df.columns]
    feats = [c for c in df.columns if c not in meta + binaries]

    df.to_parquet(OUT_DIR / f'{name}_nan.parquet', index=False)
    print(f'  saved {name}_nan.parquet   (NaN intact, {df.shape[1]} cols)')

    nan_before = df[feats].isna().sum()
    total = len(df) * len(feats)
    print(f'  residual NaN: {int(nan_before.sum()):,} of {total:,} cells '
          f'({nan_before.sum() / total:.3%})')

    out = df.copy()
    out[feats] = out[feats].clip(-CLIP, CLIP)      # binaries excluded: 0/1, no-op
    n_clipped = int((df[feats].abs() > CLIP).sum().sum())
    print(f'  clipped at +/-{CLIP:.0f}: {n_clipped:,} cells '
          f'({n_clipped / total:.3%})')

    out[feats] = out[feats].fillna(0.0)
    if binaries:
        out[binaries] = out[binaries].fillna(0.0)

    num = [c for c in out.columns if c != 'date']
    out[num] = out[num].astype(DTYPE)

    assert out[feats].isna().sum().sum() == 0, 'NaN survived the fill'
    out.to_parquet(OUT_DIR / f'{name}.parquet', index=False)
    print(f'  saved {name}.parquet   ({DTYPE}, {out.shape[1]} cols, {len(out):,} rows)')

    rep = pd.DataFrame({
        'dataset':      name,
        'feature':      feats,
        'n_nan':        nan_before.values,
        'pct_nan':      (nan_before / len(df)).values,
        'n_clipped':    (df[feats].abs() > CLIP).sum().values,
    })
    rep['flag'] = np.where(rep['pct_nan'] > 0.05, 'IMPUTED >5%', '')
    return rep

rep = pd.concat([finalise(agg_means, 'agg_means'),
                 finalise(agg_full,  'agg_full_moments')], ignore_index=True)
rep.to_csv(OUT_DIR / 'fill_report_aggregate.csv', index=False)


agg_means
  saved agg_means_nan.parquet   (NaN intact, 581 cols)
  residual NaN: 0 of 2,516,416 cells (0.000%)
  clipped at +/-5: 11,769 cells (0.468%)
  saved agg_means.parquet   (float32, 581 cols, 4,384 rows)

agg_full_moments
  saved agg_full_moments_nan.parquet   (NaN intact, 1706 cols)
  residual NaN: 0 of 7,448,416 cells (0.000%)
  clipped at +/-5: 34,899 cells (0.469%)
  saved agg_full_moments.parquet   (float32, 1706 cols, 4,384 rows)


In [4]:
print('=' * 100)
print('IMPUTED MASS - features above 5% zero-filled')
print('=' * 100)
print('After z-scoring, zero means "exactly the expanding mean", which is')
print('fabricated. Above ~5% the spline at zero is reading mostly imputation.\n')

bad = rep[rep['flag'] != ''].sort_values('pct_nan', ascending=False)
if len(bad):
    print(bad[['dataset', 'feature', 'n_nan', 'pct_nan', 'n_clipped']]
          .to_string(index=False, formatters={'pct_nan': '{:.2%}'.format}))
else:
    print('  none -- every feature below 5%')

print('\n' + '-' * 100)
print('DISTRIBUTION OF IMPUTED MASS')
print('-' * 100)
for ds, g in rep.groupby('dataset'):
    print(f'  {ds:<20} median {g["pct_nan"].median():.3%}   '
          f'p95 {g["pct_nan"].quantile(.95):.3%}   max {g["pct_nan"].max():.3%}   '
          f'features >1% {int((g["pct_nan"] > .01).sum())}')

print('\n' + '-' * 100)
print('MOST CLIPPED')
print('-' * 100)
print(rep.nlargest(15, 'n_clipped')[['dataset', 'feature', 'n_clipped', 'pct_nan']]
      .to_string(index=False, formatters={'pct_nan': '{:.2%}'.format}))

IMPUTED MASS - features above 5% zero-filled
After z-scoring, zero means "exactly the expanding mean", which is
fabricated. Above ~5% the spline at zero is reading mostly imputation.

  none -- every feature below 5%

----------------------------------------------------------------------------------------------------
DISTRIBUTION OF IMPUTED MASS
----------------------------------------------------------------------------------------------------
  agg_full_moments     median 0.000%   p95 0.000%   max 0.000%   features >1% 0
  agg_means            median 0.000%   p95 0.000%   max 0.000%   features >1% 0

----------------------------------------------------------------------------------------------------
MOST CLIPPED
----------------------------------------------------------------------------------------------------
         dataset               feature  n_clipped pct_nan
       agg_means                   Tax        313   0.00%
agg_full_moments            Tax_cwmean        313   0.00%
a